# ShopDesk, Module 2 Section 3 Lab 1: Grep, Glob, and Edit

A beginner-friendly notebook on Claude Code's **built-in code tools**. We create a small
ShopDesk **sample repo**, use **Grep** to locate functions, **Glob** to discover test and
config files, and **Edit** to make a targeted change, learning why Edit needs a **unique**
anchor. Pure-Python analogues make each tool concrete offline; a live **Claude Agent SDK** run
uses the real tools. Runs **Sonnet** (`claude-sonnet-4-6`) through your **Anthropic API key**.

## The real-world scenario

To change one line in ShopDesk's refund rule, an agent should not read the whole codebase. It
**searches** for where the rule lives (Grep), **discovers** the right files by name (Glob), then
makes a **surgical** change (Edit). Each tool does one job, and using the right one keeps the
context small and the change safe.

The question this lab answers: **which built-in tool does what, and why does a targeted Edit
require a uniquely matching anchor?**

## Objectives

- Use **Grep** (content search) to locate functions across the codebase.
- Use **Glob** (path patterns) to find test and config files.
- Use **Edit** for a targeted change, and see why an **ambiguous anchor** fails.

## What you'll observe

- Grep returns file and line for each match; Glob returns matching file paths.
- A unique-anchor Edit succeeds; an anchor that appears twice in a file is rejected.
- Live, the agent uses Grep, then Read, then Edit to change one line safely.

## How to run

Run top to bottom. The repo creation and the pure-Python tool cells run anywhere. The live cell
calls Claude with the real built-in tools, so paste a real key into **Setup 2/3** and re-run from
the top; otherwise it skips. **Node.js 18+** must be installed for the Agent SDK.

## 0. Setup

**This cell:** installs the packages. The live cell uses the **Agent SDK** to drive Claude
Code's built-in tools; the offline cells use only Python. The Agent SDK also needs Node.js 18+.

In [ ]:
# ===== SETUP 1/3 - install the Agent SDK =====
%pip install -q claude-agent-sdk anthropic python-dotenv

**This cell:** imports, the model, the `RUN_LIVE` switch, and `run_async()` for the live
cell.

In [ ]:
# ===== SETUP 2/3 - imports, the model, the switch, and an async runner =====
import os                                       # filesystem paths for the sample repo
import re                                       # our offline Grep uses regex
import sys                                       # detect Windows (it needs a special event loop)
from pathlib import Path                          # our offline Glob uses pathlib
import asyncio                                  # the Agent SDK is async; we drive it ourselves
import threading                                # run that async loop in a side thread (notebook-safe)

try:                                            # load a .env file if present
    from dotenv import load_dotenv              #   import the loader
    load_dotenv()                               #   read .env into environment variables
except Exception:                               # not installed? that is fine
    pass                                        #   set the key another way

MODEL = "claude-sonnet-4-6"                      # the Sonnet model the live cell will use

os.environ.setdefault("ANTHROPIC_API_KEY", "sk-ant-...")     # placeholder unless you set a real key
_key = os.environ["ANTHROPIC_API_KEY"]           # read whatever key is set
RUN_LIVE = _key.startswith("sk-ant-") and _key != "sk-ant-..."   # True only for a real key

def run_async(make_coro):                        # run any async Agent SDK call, notebook-safe
    box = {}                                     #   carries the result/error out of the thread
    def worker():                                #   runs in its own thread
        loop = asyncio.ProactorEventLoop() if sys.platform == "win32" else asyncio.new_event_loop()
        asyncio.set_event_loop(loop)             #     make it this thread's loop
        try:    box["value"] = loop.run_until_complete(make_coro())   # run to completion
        except Exception as e: box["error"] = e  #     capture any error
        finally: loop.close()                    #     always close the loop
    t = threading.Thread(target=worker); t.start(); t.join()   # run it and wait
    if "error" in box: raise box["error"]        #   surface any error here
    return box.get("value")                      #   hand back the result

print("live model calls:", "ON" if RUN_LIVE else "OFF (using a placeholder key)")

**This cell:** writes a small **ShopDesk sample repo** to disk: a package with orders,
refunds, and shipping modules, a couple of tests, and config files. Everything below searches and
edits these real files, so the tools have something concrete to act on.

In [ ]:
# ===== SETUP 3/3 - create the sample repo =====
import textwrap                                    # keeps the embedded file bodies readable
REPO = os.path.join(os.getcwd(), "shopdesk_repo")  # the sample codebase root

FILES = {
    "shopdesk/__init__.py": "",
    "shopdesk/orders.py": textwrap.dedent("""\
        ORDERS = {
            "A1": {"status": 2, "delivered_days_ago": 5},
            "A2": {"status": 3, "delivered_days_ago": 60},
        }
        STATUS_NAMES = {1: "processing", 2: "shipped", 3: "delivered"}

        def get_order_status(order_id):
            order = _lookup(order_id)
            if order is None:
                return None
            return STATUS_NAMES[order["status"]]

        def _lookup(order_id):
            return ORDERS.get(order_id)
        """),
    "shopdesk/refunds.py": textwrap.dedent("""\
        from shopdesk.orders import _lookup

        REFUND_WINDOW_DAYS = 30

        def is_refundable(order_id):
            order = _lookup(order_id)
            if order is None:
                return None
            return order["delivered_days_ago"] <= REFUND_WINDOW_DAYS

        def process_refund(order_id):
            eligible = is_refundable(order_id)
            if eligible is None:
                return None
            return "refunded " + order_id if eligible else "refused: outside window"
        """),
    "shopdesk/shipping.py": textwrap.dedent("""\
        def get_tracking(order_id):
            return "https://track.example/" + order_id
        """),
    "tests/test_orders.py": textwrap.dedent("""\
        from shopdesk.orders import get_order_status
        def test_status():
            assert get_order_status("A1") == "shipped"
        """),
    "tests/test_refunds.py": textwrap.dedent("""\
        from shopdesk.refunds import process_refund
        def test_refund():
            assert process_refund("A1").startswith("refunded")
        """),
    "config/settings.toml": '[shopdesk]\nrefund_window_days = 30\n',
    "pyproject.toml": '[tool.pytest.ini_options]\ntestpaths = ["tests"]\n',
    "README.md": "# ShopDesk sample repo\n",
}
for rel, content in FILES.items():                 # write every file
    path = os.path.join(REPO, rel)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        f.write(content)
print("created repo at", REPO, "with", len(FILES), "files")

### Four tools, four jobs

- **Grep** searches file *contents* by pattern (ripgrep under the hood). "Which files define
  `process_refund`?" is a Grep.
- **Glob** searches file *paths* by pattern. "Where are the tests?" is a Glob (`**/test_*.py`).
- **Read** loads one file (or a slice), with line numbers.
- **Edit** makes a targeted change by replacing a uniquely matching anchor; **Write** overwrites a
  whole file.

The rule that trips people up: **Edit's anchor must be unique in the file.** If the text appears
more than once, Edit fails rather than risk changing the wrong one.

---

### 🎯 Lab objective - search, discover, edit

**What you build:** offline analogues of Grep, Glob, and Edit over the sample repo, plus a live run
that uses the real tools.

**Why it helps you build real solutions:** picking the right tool (Grep vs Glob) and respecting
Edit's uniqueness rule is what keeps an agent's changes small, correct, and cheap.

**How you'll see it:** Grep and Glob find exactly what you ask for, and Edit accepts a unique anchor
while rejecting an ambiguous one.

**This cell:** an offline **Glob**: match files by path pattern. We find the test files with
`**/test_*.py` and the config files with `**/*.toml`. Glob answers "which files exist" without ever
opening them.

In [ ]:
# ===== Glob: find files by path pattern =====
def glob_files(pattern):                           # pattern -> sorted matching paths (relative)
    return sorted(str(p.relative_to(REPO)) for p in Path(REPO).glob(pattern) if p.is_file())

print("tests  :", glob_files("**/test_*.py"))      # discover the test files
print("configs:", glob_files("**/*.toml"))         # discover the config files

**This cell:** an offline **Grep**: search file *contents* for a pattern and report the file
and line of every match. We locate the `process_refund` function and everywhere the refund window
constant is used. Grep answers "where does this live" across the codebase.

In [ ]:
# ===== Grep: find content across files =====
def grep(pattern, file_glob="**/*.py"):            # pattern -> list of (path, line_no, text)
    rx = re.compile(pattern)                        #   ripgrep-style regex (Python re here)
    hits = []
    for p in Path(REPO).glob(file_glob):            #   scan matching files
        if not p.is_file():
            continue
        for i, line in enumerate(p.read_text().splitlines(), 1):   # line by line
            if rx.search(line):
                hits.append((str(p.relative_to(REPO)), i, line.strip()))
    return hits

for hit in grep(r"def process_refund"):            # locate the function definition
    print("  ", hit)
for hit in grep(r"REFUND_WINDOW_DAYS"):            # every use of the constant
    print("  ", hit)

**This cell:** a small offline **Read** that returns a slice of a file with line numbers,
mirroring how Read works. We will need it before editing, because Edit requires you to have read the
file first (the read-before-edit rule).

In [ ]:
# ===== Read: load a file (or a slice) with line numbers =====
def read_file(rel, start=1, end=None):             # rel path -> numbered lines from start..end
    lines = open(os.path.join(REPO, rel)).read().splitlines()
    end = end or len(lines)
    return "\n".join(f"{i:4}  {lines[i-1]}" for i in range(start, min(end, len(lines)) + 1))

print(read_file("shopdesk/refunds.py", 1, 4))      # read the top of the refunds module

**This cell:** an offline **Edit** that enforces the real rule: the anchor must appear exactly
once. It returns a clear failure for a missing anchor and for an **ambiguous** one (more than one
match), instead of guessing. This is the safety guarantee behind Edit.

In [ ]:
# ===== Edit: replace a UNIQUE anchor (fails on ambiguity) =====
def edit(rel, old, new, replace_all=False):        # targeted replacement with the uniqueness rule
    path = os.path.join(REPO, rel)
    text = open(path).read()
    n = text.count(old)                             #   how many times the anchor appears
    if n == 0:
        return f"FAILED: anchor not found in {rel}"
    if n > 1 and not replace_all:                   #   ambiguous -> refuse (the key rule)
        return f"FAILED: anchor appears {n} times in {rel}; add context or use replace_all"
    open(path, "w").write(text.replace(old, new))
    return f"OK: replaced {n if replace_all else 1} occurrence in {rel}"

**This cell:** a **unique-anchor** edit that succeeds. `REFUND_WINDOW_DAYS = 30` appears exactly
once in `refunds.py`, so Edit can change the window from 30 to 45 days safely.

In [ ]:
# ===== a unique anchor -> Edit succeeds =====
print(edit("shopdesk/refunds.py", "REFUND_WINDOW_DAYS = 30", "REFUND_WINDOW_DAYS = 45"))
print(read_file("shopdesk/refunds.py", 3, 3))      # confirm the change

**This cell:** an **ambiguous-anchor** edit that fails. The line `return None` appears twice in
`refunds.py`, so Edit refuses rather than risk changing the wrong one. This is exactly the failure the
next lab's Read+Write fallback is for.

In [ ]:
# ===== an ambiguous anchor -> Edit refuses =====
print("occurrences of 'return None':", open(os.path.join(REPO, "shopdesk/refunds.py")).read().count("return None"))
print(edit("shopdesk/refunds.py", "return None", 'return "unknown order"'))   # -> FAILED (2 matches)

**This cell:** the live run using the **real built-in tools**. We point the agent at the sample
repo with `cwd=REPO` and allow only `Grep`, `Glob`, `Read`, and `Edit`. Ask it to change the window,
and watch it Grep to find the line, Read the file, then Edit, exactly the workflow above.

In [ ]:
# ===== live: let Claude Code use Grep / Glob / Read / Edit =====
from claude_agent_sdk import query, ClaudeAgentOptions, AssistantMessage, TextBlock, ToolUseBlock

CODE_OPTS = ClaudeAgentOptions(                     # point the tools at our sample repo
    model=MODEL, cwd=REPO,
    allowed_tools=["Grep", "Glob", "Read", "Edit"])

async def ask(prompt):                              # stream the run and show each tool call
    async for m in query(prompt=prompt, options=CODE_OPTS):
        if isinstance(m, AssistantMessage):
            for b in m.content:
                if isinstance(b, ToolUseBlock): print("  ->", b.name, {k: b.input[k] for k in list(b.input)[:2]})
                elif isinstance(b, TextBlock) and b.text.strip(): print("  ", b.text.strip()[:140])

if RUN_LIVE:                                        # needs a real key (and Node.js 18+)
    run_async(lambda: ask("In shopdesk/refunds.py change the refund window to 60 days. Grep for it, Read the file, then Edit."))
else:
    print("[skipped] expected: Grep finds REFUND_WINDOW_DAYS, Read loads refunds.py, Edit sets it to 60.")

| anti-pattern | what to do instead |
|---|---|
| Grep for a filename | use Glob for paths, Grep for contents |
| Edit with a short, common anchor | include enough surrounding context to make it unique |
| Edit without reading first | Read the file first (read-before-edit) |
| Write the whole file for a one-line change | use Edit for surgical changes |

**Lesson:** Claude Code's tools each do one job: **Grep** finds content, **Glob** finds files,
**Read** loads one file, **Edit** changes a unique anchor, **Write** replaces a whole file. Edit's
uniqueness rule is a feature, not a limitation: it stops the agent from changing the wrong line. When
an anchor cannot be made unique, that is the signal to fall back to Read plus Write, which is the next
lab.

---

## Recap - the built-in code tools

| Tool | Finds / does | Use for |
|---|---|---|
| Grep | content by pattern | locating functions and usages |
| Glob | file paths by pattern | discovering tests and config |
| Read | one file (or slice) | inspecting before editing |
| Edit | replace a unique anchor | targeted, surgical changes |

One principle to carry forward: **search first, read the slice you need, then make the smallest safe
change.** To run live, paste a real key into **Setup 2/3** and re-run from the top. Then try it: ask
the agent to rename `is_refundable` and watch Edit need extra context to stay unique. Next lab: the
Read plus Write fallback and incremental exploration.